<a href="https://colab.research.google.com/github/mandaw34/BigData26_A_2411533009_Devina-Amanda-P/blob/main/Praktium%205/BD_P05_2411533009_Devina_Amanda_Putri.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install -q duckdb beautifulsoup4 lxml requests pyarrow pandas

**H-2. Persiapan: Unduh Data dan Jalankan Portal Lokal**

Cell ini berfungsi untuk menyiapkan lingkungan data lokal dengan mengunduh dataset mentah berupa file CSV berisi judul berita Indonesia dari repositori GitHub. Setelah file tersedia, fungsi jalankan_portal menggunakan library http.server untuk menghidupkan server web lokal portabel pada alamat 127.0.0.1:8000. Server mini ini mensimulasikan sebuah portal berita nyata yang memecah ribuan data CSV menjadi halaman-halaman web HTML terstruktur (/berita?hal=N) serta menyediakan file robots.txt untuk kebutuhan simulasi crawling yang etis dan terkontrol tanpa membebani server publik

In [1]:
import os, re, json, time, zlib, queue, threading, collections
from pathlib import Path
import urllib.request
import pandas as pd

URL_DATA  = ("https://raw.githubusercontent.com/ibamibrahim/"
"dataset-judul-berita-indonesia/master/detik_news_title.csv")
if not os.path.exists("detik_news_title.csv"):
    urllib.request.urlretrieve(URL_DATA, "detik_news_title.csv")

df_sumber = pd.read_csv("detik_news_title.csv")
print("Ukuran berkas:", round(os.path.getsize("detik_news_title.csv") / 1e6, 1), "MB")
print("Baris, kolom:", df_sumber.shape)
print("Kolom:", list(df_sumber.columns))

import http.server, urllib.parse, html
HALAMAN_UKURAN = 100 # judul per halaman

def jalankan_portal(df, port=8000):
    """Portal berita lokal (siap pakai): menyajikan judul dalam halaman HTML."""
    baris = df.to_dict("records")
    n_hal = (len(baris) + HALAMAN_UKURAN - 1) // HALAMAN_UKURAN

    class Handler(http.server.BaseHTTPRequestHandler):
        def log_message(self, *args):
            pass # matikan log

        def _kirim(self, kode, teks, tipe="text/html; charset=utf-8"):
            data = teks.encode("utf-8")
            self.send_response(kode)
            self.send_header("Content-Type", tipe)
            self.send_header("Content-Length", str(len(data)))
            self.end_headers()
            self.wfile.write(data)

        def do_GET(self):
            u = urllib.parse.urlparse(self.path)
            if u.path == "/robots.txt":
                self._kirim(200, "User-agent: *\nDisallow: /arsip-privat/\n", "text/plain")
            elif u.path == "/arsip-privat/":
                self._kirim(200, "<p>Halaman ini tidak boleh di-crawl.</p>")
            elif u.path == "/berita":
                q = urllib.parse.parse_qs(u.query)
                hal = int(q.get("hal", ["1"])[0])

                # PERBAIKAN DI SINI: Ubah dari 1 < hal menjadi 1 <= hal agar halaman 1 bisa diakses
                if not 1 <= hal <= n_hal:
                    return self._kirim(404, "<p>Halaman tidak ada</p>")

                potong = baris[(hal - 1) * HALAMAN_UKURAN : hal * HALAMAN_UKURAN]
                isi = []
                for r in potong:
                    iso = pd.to_datetime(r["date"], format="%m/%d/%Y").strftime("%Y-%m-%d") if pd.notna(r["date"]) else ""
                    isi.append(f'<article class="item"><h2 class="judul"><a href="{html.escape(str(r["url"]))}">{html.escape(str(r["title"]))}</a></h2><span class="kategori">{html.escape(str(r["category"]))}</span><time datetime="{iso}">{html.escape(str(r["date"]))}</time></article>')
                self._kirim(200, f"<html><body><h1>Portal Contoh</h1>{''.join(isi)}<nav data-total=\"{n_hal}\"></nav></body></html>")
            else:
                self._kirim(404, "<p>Tidak ditemukan</p>")

    server = http.server.ThreadingHTTPServer(("127.0.0.1", port), Handler)
    threading.Thread(target=server.serve_forever, daemon=True).start()
    return server, n_hal

server, N_HALAMAN = jalankan_portal(df_sumber)
BASE = "http://127.0.0.1:8000"
print("Portal lokal aktif di", BASE, "| jumlah halaman:", N_HALAMAN)

Ukuran berkas: 16.1 MB
Baris, kolom: (91017, 4)
Kolom: ['date', 'url', 'title', 'category']
Portal lokal aktif di http://127.0.0.1:8000 | jumlah halaman: 911


**H-3. Crawling Portal Lokal: Latihan Aman**

Tahap ini melatih mekanisme crawling secara etis dan bertanggung jawab dengan memanfaatkan server lokal. Fungsi baca_robots mengunduh dan mengurai file robots.txt menggunakan urllib.robotparser untuk memeriksa apakah suatu direktori diizinkan untuk diakses oleh user-agent tertentu. Selanjutnya, fungsi parse_halaman menggunakan BeautifulSoup dengan pemilih CSS (article.item, h2.judul a, dll.) untuk mengekstrak informasi terstruktur dari dokumen HTML. Fungsi generator crawl_iter mengatur iterasi pengambilan halaman secara aman dengan menyertakan mekanisme penanganan kesalahan (try-except dengan retry), jeda waktu (delay), serta kepatuhan penuh terhadap aturan robots.txt sebelum menghasilkan data secara streaming

In [3]:
import requests, urllib.robotparser
from bs4 import BeautifulSoup

UA = "KelasBigData-UNAND/1.0 (praktikum; kontak: dosen)"

def baca_robots(base):
 r = requests.get(base + "/robots.txt", headers={"User-Agent": UA}, timeout=10)
 rp = urllib.robotparser.RobotFileParser()
 rp.parse(r.text.splitlines())
 return rp

robots = baca_robots(BASE)
for jalur in ["/berita?hal=1", "/arsip-privat/"]:
 boleh = robots.can_fetch(UA, BASE + jalur)
 print(f"{jalur:<18} boleh di-crawl? {boleh}")

/berita?hal=1      boleh di-crawl? True
/arsip-privat/     boleh di-crawl? False


In [4]:
import requests, urllib.robotparser
from bs4 import BeautifulSoup

UA = "KelasBigData-UNAND/1.0 (praktikum; kontak: dosen)"

def baca_robots(base):
    r = requests.get(base + "/robots.txt", headers={"User-Agent": UA}, timeout=10)
    rp = urllib.robotparser.RobotFileParser()
    rp.parse(r.text.splitlines())
    return rp

robots = baca_robots(BASE)
for jalur in ["/berita?hal=1", "/arsip-privat/"]:
    boleh = robots.can_fetch(UA, BASE + jalur)
    print(f"{jalur: <18} boleh di-crawl? {boleh}")

def parse_halaman(teks_html):
    """Ubah HTML satu halaman menjadi daftar dict (judul, url, kategori, tanggal)."""
    sup = BeautifulSoup(teks_html, "html.parser")
    hasil = []
    for a in sup.select("article.item"):
        link = a.select_one("h2.judul a")
        hasil.append({
            "judul": link.get_text(strip=True),
            "url": link["href"],
            "kategori": a.select_one("span.kategori").get_text(strip=True),
            "tanggal": a.select_one("time")["datetime"],
        })
    return hasil

def crawl_iter(base, halaman, delay=0.0, ua=UA):
    """Siap pakai: ambil halaman satu per satu, patuhi robots.txt, beri jeda."""
    rp = baca_robots(base)
    sesi = requests.Session()
    sesi.headers["User-Agent"] = ua
    for h in halaman:
        url = f"{base}/berita?hal={h}"
        if not rp.can_fetch(ua, url):
            print("Dilewati (robots.txt):", url)
            continue
        for percobaan in range(3):
            try:
                r = sesi.get(url, timeout=10)
                r.raise_for_status()
                break
            except requests.RequestException as e:
                time.sleep(0.5 * (percobaan + 1))
        else:
            print("Gagal setelah 3 percobaan:", url)
            continue
        yield from parse_halaman(r.text)
        time.sleep(delay)

contoh = list(crawl_iter(BASE, range(1, 4)))
print("Judul dari 3 halaman:", len(contoh))
for c in contoh[:3]:
    print(c)

/berita?hal=1      boleh di-crawl? True
/arsip-privat/     boleh di-crawl? False
Judul dari 3 halaman: 300
{'judul': 'Kemnaker Awasi TKA di Meikarta', 'url': 'https://finance.detik.com/berita-ekonomi-bisnis/d-4916114/kemnaker-awasi-tka-di-meikarta', 'kategori': 'finance', 'tanggal': '2020-02-26'}
{'judul': 'BNI Digitalkan BNI Java Jazz 2020', 'url': 'https://finance.detik.com/berita-ekonomi-bisnis/d-4916199/bni-digitalkan-bni-java-jazz-2020', 'kategori': 'finance', 'tanggal': '2020-02-26'}
{'judul': 'Terbang ke Australia, Edhy Prabowo Mau Genjot Budi Daya Lobster', 'url': 'https://finance.detik.com/berita-ekonomi-bisnis/d-4916193/terbang-ke-australia-edhy-prabowo-mau-genjot-budi-daya-lobster', 'kategori': 'finance', 'tanggal': '2020-02-26'}


**H-4. Crawling Berita Sungguhan dari Sitemap detik.com**

Implementasikan pengambilan data langsung dari sumber eksternal nyata menggunakan file Sitemap XML situs berita detik.com alih-alih merayapi halaman web satu per satu. BeautifulSoup dikonfigurasi dengan parser "xml" bersama pustaka lxml untuk membaca struktur namespace berita (news:title, news:publication_date, dll.) secara efisien. Kode ini dilengkapi sistem pengamanan ketat meliputi pengecekan robots.txt independen per hos, pembatasan jumlah permintaan (BATAS_TOTAL dan jeda 2 detik), identifikasi User-Agent yang jelas, serta proteksi terhadap respons pembatasan akses server (403 atau 429)

langkah 1 - tulis parser

In [5]:
from bs4 import BeautifulSoup
from urllib.parse import urlparse

POLA_ARTIKEL = re.compile(r"https?://([\w-]+)\.detik\.com/.+/d-(\d+)/")

def parse_sitemap_news(xml_data):
    """Ubah sitemap berita (XML) menjadi daftar dict (judul, url, kategori, id, tanggal, kata_kunci)."""
    sup = BeautifulSoup(xml_data, "xml")
    hasil = []
    for u in sup.find_all("url"):
        loc, judul, waktu = u.find("loc"), u.find("title"), u.find("publication_date")
        if loc is None or judul is None:
            continue
        url = loc.get_text(strip=True)
        m = POLA_ARTIKEL.match(url)
        kunci = u.find("keywords")
        hasil.append({
            "judul": re.sub(r"\s+", " ", judul.get_text(strip=True)),
            "url": url,
            "kategori": m.group(1) if m else None, # news, inet, sport, ...
            "id": int(m.group(2)) if m else None,
            "tanggal": waktu.get_text(strip=True)[:10] if waktu else None,
            "kata_kunci": kunci.get_text(strip=True) if kunci else ""
        })
    return hasil

Langklah 2 - uji tanpa internet

In [6]:
CONTOH_XML = """<?xml version="1.0" encoding="UTF-8"?>
<urlset xmlns="http://www.sitemaps.org/schemas/sitemap/0.9"
        xmlns:news="http://www.google.com/schemas/sitemap-news/0.9">
<url>
<loc><![CDATA[https://news.detik.com/berita/d-8729282/mk-kepala-daerah-terdakwa-politik-tak-bisa-diberhentikan-tanpa-usulan-dprd]]></loc>
<news:news>
<news:publication><news:name>detikNews</news:name><news:language>id</news:language></news:publication>
<news:publication_date>2026-10-29T12:52:00+00:00</news:publication_date>
<news:title><![CDATA[MK: Kepala Daerah 'Terdakwa Politik' Tak Bisa Diberhentikan Tanpa Usulan DPRD]]></news:title>
<news:keywords><![CDATA[mahkamah konstitusi, mk, hukum, kepala daerah]]></news:keywords>
</news:news>
</url>
<url>
<loc><![CDATA[https://news.detik.com/berita/d-8729280/tep-unair-periksa-1-000-sapi-di-manokwari-lewat-posyandu-ternak-keliling]]></loc>
<news:news>
<news:publication_date>2026-10-29T12:50:00+00:00</news:publication_date>
<news:title><![CDATA[TEP UNAIR Periksa 1.000 Sapi di Manokwari Lewat Posyandu Ternak Keliling]]></news:title>
</news:news>
</url>
<url>
<loc><![CDATA[https://news.detik.com/berita/d-8729264/andre-rosiade-dorong-sedimen-sungai-akibat-banjir-jadi-sumber-pendapatan-padang]]></loc>
<news:news>
<news:publication_date>2026-10-29T12:44:00+00:00</news:publication_date>
<news:title><![CDATA[Andre Rosiade Dorong Sedimen Sungai Akibat Banjir Jadi Sumber Pendapatan Padang]]></news:title>
</news:news>
</url>
</urlset>"""

for x in parse_sitemap_news(CONTOH_XML):
    print({k: x[k] for k in ("judul", "kategori", "id", "tanggal")})
print("kata kunci entri pertama:", parse_sitemap_news(CONTOH_XML)[0]["kata_kunci"])
print("Sama bila masukan berupa bytes:", parse_sitemap_news(CONTOH_XML.encode("utf-8")) == parse_sitemap_news(CONTOH_XML))

{'judul': "MK: Kepala Daerah 'Terdakwa Politik' Tak Bisa Diberhentikan Tanpa Usulan DPRD", 'kategori': 'news', 'id': 8729282, 'tanggal': '2026-10-29'}
{'judul': 'TEP UNAIR Periksa 1.000 Sapi di Manokwari Lewat Posyandu Ternak Keliling', 'kategori': 'news', 'id': 8729280, 'tanggal': '2026-10-29'}
{'judul': 'Andre Rosiade Dorong Sedimen Sungai Akibat Banjir Jadi Sumber Pendapatan Padang', 'kategori': 'news', 'id': 8729264, 'tanggal': '2026-10-29'}
kata kunci entri pertama: mahkamah konstitusi, mk, hukum, kepala daerah
Sama bila masukan berupa bytes: True


langkah 3 - fungsi pengambilan yang sopan

In [7]:
import urllib.robotparser

UA_DETIK = "BigData/1.0 (praktikum pembelajaran; kontak: GANTI_DENGAN_EMAIL_ANDA)"
DELAY = 2.0               # detik antar-request. Jangan dikurangi.
BATAS_PER_BAGIAN = 7      # judul yang diambil dari tiap sitemap
BATAS_TOTAL = 20          # batas seluruh crawling
SITEMAP_DIPILIH = [
    "https://news.detik.com/berita/sitemap_news.xml",
    "https://inet.detik.com/news/sitemap_news.xml",
    "https://sport.detik.com/basket/sitemap_news.xml",
]

_robots = {}
def boleh_diambil(url, ua=UA_DETIK):
    """robots.txt berlaku per host: news.detik.com dan inet.detik.com punya berkas sendiri."""
    host = urlparse(url).netloc
    if host not in _robots:
        rp = urllib.robotparser.RobotFileParser()
        rp.set_url(f"https://{host}/robots.txt")
        rp.read()
        _robots[host] = rp
    return _robots[host].can_fetch(ua, url)

def ambil_xml(url, sesi, delay=DELAY):
    if not boleh_diambil(url):
        raise PermissionError("robots.txt melarang: " + url)
    r = sesi.get(url, timeout=20)
    if r.status_code in (403, 429):
        raise RuntimeError(f"Server membalas {r.status_code}. Berhenti, tunggu, dan kurangi laju.")
    r.raise_for_status()
    time.sleep(delay)
    return r.content # bytes: pengkodean dibaca dari deklarasi XML

def crawl_detik():
    sesi = requests.Session()
    sesi.headers["User-Agent"] = UA_DETIK
    hasil = []
    for sm in SITEMAP_DIPILIH:
        if len(hasil) >= BATAS_TOTAL:
            break
        hasil += parse_sitemap_news(ambil_xml(sm, sesi))[:BATAS_PER_BAGIAN]
    return hasil[:BATAS_TOTAL]

langkah 4 - jalankan dan periksa

In [8]:
try:
    sampel = crawl_detik()
except Exception as e:
    sampel = []
    print("Crawling gagal:", type(e).__name__, "-", e)

if sampel:
    Path("contoh_crawl").mkdir(exist_ok=True)
    with open("contoh_crawl/detik_sitemap.jsonl", "w", encoding="utf-8") as f:
        for x in sampel:
            f.write(json.dumps(x, ensure_ascii=False) + "\n")
    cek = pd.DataFrame(sampel)
    print("Judul terkumpul:", len(cek))
    print("Judul kosong:", (cek["judul"].str.strip() == "").sum())
    print("URL duplikat:", cek["url"].duplicated().sum())
    print("Tanggal kosong:", cek["tanggal"].isna().sum())
    print("Kategori:", cek["kategori"].value_counts().to_dict())
    print(cek[["judul", "kategori", "tanggal"]].head(3).to_string(index=False))

Judul terkumpul: 20
Judul kosong: 0
URL duplikat: 0
Tanggal kosong: 0
Kategori: {'news': 7, 'inet': 7, 'sport': 6}
                                                                        judul kategori    tanggal
Kronologi BNN-Polri Tangkap Muzammil DPO Kasus Narkoba Jaringan Internasional     news 2026-11-12
            Respons KPK Usai Disebut Kapolri Jadi Korban Penyadapan Listyo Cs     news 2026-11-12
        Di Forum COP31 Turki, Hashim Bicara Upaya Pemerintah Tangani Karhutla     news 2026-11-12


**H-5. Channel ala Flume ke lapisan_mentah**

Arsitektur penanganan data berskala penuh disimulasikan menggunakan pola Source-Channel-Sink terinspirasi dari kerangka kerja Apache Flume. Fungsi source berjalan pada thread terpisah untuk mengekstrak data dari portal lokal dan memasukkannya ke dalam antrean terbatas (queue.Queue dengan kapasitas maksimum 2000 elemen untuk menerapkan mekanisme backpressure). Di sisi lain, fungsi sink mengambil data dari antrean secara asinkron dan menulisnya dalam format JSON baris demi baris (.jsonl) ke dalam folder lapisan_mentah/ setiap terkumpul 5.000 data per batch.

In [9]:
RAW = Path("lapisan_mentah")
RAW.mkdir(exist_ok=True)
for f in RAW.glob("*.jsonl"):
    f.unlink()

channel = queue.Queue(maxsize=2000)
STOP = object()
statistik = {"max_antrean": 0, "masuk": 0, "keluar": 0, "file": 0}

def source():
    for ev in crawl_iter(BASE, range(1, N_HALAMAN + 1)):
        channel.put(ev)  # menunggu bila channel penuh
        statistik["masuk"] += 1
        statistik["max_antrean"] = max(statistik["max_antrean"], channel.qsize())
    channel.put(STOP)

def sink(ukuran_batch=5000):
    batch, nomor = [], 0
    def tulis():
        nonlocal batch, nomor
        nomor += 1
        with open(RAW / f"batch_{nomor:04d}.jsonl", "w", encoding="utf-8") as f:
            for ev in batch:
                f.write(json.dumps(ev, ensure_ascii=False) + "\n")
        statistik["file"] = nomor
        batch = []

    while True:
        ev = channel.get()
        if ev is STOP:
            break
        batch.append(ev)
        statistik["keluar"] += 1
        if len(batch) >= ukuran_batch:
            tulis()
    if batch:
        tulis()

t0 = time.perf_counter()
t_src = threading.Thread(target=source)
t_snk = threading.Thread(target=sink)
t_src.start(); t_snk.start()
t_src.join(); t_snk.join()
waktu = time.perf_counter() - t0

print("Event masuk channel:", f"{statistik['masuk']:,}")
print("Event ditulis sink:", f"{statistik['keluar']:,}")
print("File batch:", statistik["file"], "->", sorted(p.name for p in RAW.glob('*.jsonl'))[:3], "...")
print("Antrean maks <= 2000:", statistik["max_antrean"] <= 2000)
print("Total ukuran mentah:", round(sum(p.stat().st_size for p in RAW.glob('*.jsonl')) / 1e6, 1), "MB")
print("Waktu crawl total:", round(waktu, 1), "detik untuk", N_HALAMAN, "halaman")

Event masuk channel: 91,017
Event ditulis sink: 91,017
File batch: 19 -> ['batch_0001.jsonl', 'batch_0002.jsonl', 'batch_0003.jsonl'] ...
Antrean maks <= 2000: True
Total ukuran mentah: 20.7 MB
Waktu crawl total: 54.4 detik untuk 911 halaman


**H-6. Quality, Cleaning, dan Parquet**

Cell ini menjalankan tahapan verifikasi kualitas data (data quality checks) dan pembersihan (cleaning) pada data mentah dengan menggunakan ekspresi boolean Pandas. Melalui fungsi aturan_kualitas, kita mengevaluasi anomali seperti judul kosong, duplikasi, rentang panjang kata, validitas format tanggal, hingga integritas URL artikel. Data yang telah dibersihkan kemudian disimpan ke dalam format kolom terkompresi Apache Parquet dengan skema partisi hierarkis berbasis Hive (partition_cols=["kategori"]), yang memungkinkan efisiensi pembacaan data tingkat lanjut melalui mekanisme partition pruning

In [10]:
mentah = pd.concat([pd.read_json(p, lines=True) for p in sorted(RAW.glob("*.jsonl"))], ignore_index=True)
print("Baris mentah:", f"{len(mentah):,}")

KATEGORI_DIKENAL = {"finance", "food", "health", "hot", "inet", "news", "oto", "sport", "travel"}

def aturan_kualitas(d):
    """Setiap aturan menghasilkan Series True/False per baris."""
    jml_kata = d["judul"].fillna("").str.split().str.len()
    return {
        "judul tidak kosong": d["judul"].fillna("").str.strip() != "",
        "judul 3-30 kata": jml_kata.between(3, 30),
        "kategori dikenal": d["kategori"].isin(KATEGORI_DIKENAL),
        "tanggal valid": pd.to_datetime(d["tanggal"], format="%Y-%m-%d", errors="coerce").notna(),
        "url artikel valid": d["url"].str.contains(r"/d-\d+", regex=True, na=False),
        "judul tidak duplikat": ~d["judul"].duplicated(keep=False),
    }

for nama, lulus in aturan_kualitas(mentah).items():
    print(f"  {nama:<24} gagal: {(~lulus).sum():>6,} ({(~lulus).mean()*100:5.2f}%)")

# Proses Pembersihan Data
bersih = mentah.copy()
bersih["judul"] = bersih["judul"].str.replace(r"\s+", " ", regex=True).str.strip()
bersih = bersih.drop_duplicates(subset="judul", keep="first")
bersih = bersih[bersih["judul"].str.split().str.len().between(3, 30)].copy()
bersih["tanggal"] = pd.to_datetime(bersih["tanggal"], format="%Y-%m-%d")
bersih["bulan"] = bersih["tanggal"].dt.strftime("%Y-%m")
bersih["id"] = bersih["url"].str.extract(r"/d-(\d+)")[0].astype("Int64")
bersih["url"] = bersih["url"].where(bersih["url"].str.contains(r"/d-\d+", regex=True)) # kosong bila url rusak

print(f"Baris setelah dibersihkan: {len(bersih):,} | dibuang: {len(mentah) - len(bersih):,}")

# Menyimpan Parquet Berpartisi
import shutil, pyarrow.dataset as ds

def simpan_terpartisi(data, kolom_partisi, folder):
    shutil.rmtree(folder, ignore_errors=True)
    data.to_parquet(folder, partition_cols=kolom_partisi, index=False)
    n = len(list(Path(folder).rglob("*.parquet")))
    ukuran = sum(p.stat().st_size for p in Path(folder).rglob("*.parquet")) / 1e6
    return n, ukuran

# Perbandingan pilihan kolom partisi
for kolom in (["kategori"], ["bulan"], ["kategori", "bulan"]):
    n, mb = simpan_terpartisi(bersih, kolom, "uji_partisi")
    print(f"partisi {str(kolom):<22} -> {n:>3} file, {mb:5.1f} MB")
shutil.rmtree("uji_partisi")

TERKURASI = "lapisan_terkurasi"
n, mb = simpan_terpartisi(bersih, ["kategori"], TERKURASI)
print("\nDisimpan ke", TERKURASI, "dengan partisi kategori:", n, "file,", round(mb, 1), "MB")

dataset = ds.dataset(TERKURASI, format="parquet", partitioning="hive")
semua = list(dataset.get_fragments())
hanya = list(dataset.get_fragments(filter=ds.field("kategori") == "finance"))
print("Partition pruning: baca", len(hanya), "dari", len(semua), "file untuk kategori finance")

Baris mentah: 91,017
  judul tidak kosong       gagal:      0 ( 0.00%)
  judul 3-30 kata          gagal:     78 ( 0.09%)
  kategori dikenal         gagal:      0 ( 0.00%)
  tanggal valid            gagal:      0 ( 0.00%)
  url artikel valid        gagal:  5,640 ( 6.20%)
  judul tidak duplikat     gagal:    235 ( 0.26%)
Baris setelah dibersihkan: 90,818 | dibuang: 199
partisi ['kategori']           ->   9 file,   8.4 MB
partisi ['bulan']              ->   6 file,   8.3 MB
partisi ['kategori', 'bulan']  ->  54 file,   8.4 MB

Disimpan ke lapisan_terkurasi dengan partisi kategori: 9 file, 8.4 MB
Partition pruning: baca 1 dari 9 file untuk kategori finance


**H-7. Query DuckDB dan Bentuk Dokumen**

Penerapan in-memory analytical database DuckDB digunakan untuk mengeksekusi kueri SQL berperforma tinggi secara langsung pada direktori file Parquet terpartisi tanpa harus memuat seluruh dataset ke dalam memori RAM Pandas. Selain agregasi kelompok data berbasis SQL (seperti distribusi kategori berita dan tren bulanan kata kunci tertentu), cell ini juga mendemonstrasikan transformasi data relasional menjadi bentuk dokumen terstruktur fleksibel (format JSON) yang memuat token kata di dalam satu record dokumen tunggal.

In [11]:
import duckdb

con = duckdb.connect()
con.sql(f"CREATE VIEW berita AS SELECT * FROM read_parquet('{TERKURASI}/*/*.parquet', hive_partitioning=true)")

print(con.sql("SELECT kategori, COUNT(*) AS jumlah FROM berita GROUP BY kategori ORDER BY jumlah DESC").df().to_string(index=False))
print()
print(con.sql("""
SELECT bulan, COUNT(*) AS judul_corona
FROM berita WHERE lower(judul) LIKE '%corona%'
GROUP BY bulan ORDER BY bulan""").df().to_string(index=False))

dokumen = {
    "id": int(bersih.iloc[0]["id"]),
    "judul": bersih.iloc[0]["judul"],
    "kategori": bersih.iloc[0]["kategori"],
    "tanggal": str(bersih.iloc[0]["tanggal"].date()),
    "kata": re.findall(r"[a-z]+", bersih.iloc[0]["judul"].lower()),
}
print()
print(json.dumps(dokumen, ensure_ascii=False, indent=2))

kategori  jumlah
    news   32321
     hot   16290
 finance   14141
  travel    6453
    inet    5577
  health    4914
     oto    4375
    food    4314
   sport    2433

  bulan  judul_corona
2020-01           385
2020-02          1077
2020-03          6994
2020-04          4913
2020-05          2934
2020-06           943

{
  "id": 4916114,
  "judul": "Kemnaker Awasi TKA di Meikarta",
  "kategori": "finance",
  "tanggal": "2020-02-26",
  "kata": [
    "kemnaker",
    "awasi",
    "tka",
    "di",
    "meikarta"
  ]
}


**H-8. Mini MapReduce: kasus kantin**

Simulasi konseptual komputasi terdistribusi MapReduce dilakukan melalui kasus transaksi kantin sederhana. Proses dibagi menjadi tiga tahap utama: Map (memetakan data transaksi per kasir menjadi pasangan kunci-nilai berupa (menu, jumlah)), Shuffle (mengelompokkan seluruh nilai berdasarkan kesamaan kunci menu), dan Reduce (menjumlahkan total porsi dari setiap kelompok kunci). Selain itu, diperkenalkan fungsi Combiner sebagai optimasi lokal sebelum data dikirim melalui jaringan, yang terbukti secara efektif memangkas volume pasangan data yang ditransmisikan

In [12]:
kasir = [
    ["nasi goreng, 1", "es teh, 2", "mie ayam, 1", "es teh, 1"],  # kasir A
    ["nasi goreng, 2", "es teh, 1", "es teh, 3", "mie ayam, 1"],  # kasir B
    ["mie ayam, 2", "nasi goreng, 1", "es teh, 1", "nasi goreng, 1"], # kasir C
]

def map_fn(baris):
    menu, jumlah = baris.split(",")
    return (menu, int(jumlah))

def reduce_fn(menu, daftar_nilai):
    return (menu, sum(daftar_nilai))

# 1) MAP
hasil_map = [[map_fn(b) for b in k] for k in kasir]
for nama, h in zip("ABC", hasil_map):
    print("map kasir", nama, "->", h)

# 2) SHUFFLE
shuffle = {}
for h in hasil_map:
    for menu, jumlah in h:
        shuffle.setdefault(menu, []).append(jumlah)
print("\nshuffle:", shuffle)

# 3) REDUCE
hasil = dict(reduce_fn(m, v) for m, v in shuffle.items())
print("\nreduce:", hasil, "| total porsi:", sum(hasil.values()))

# COMBINER
def combiner(pasangan):
    c = collections.Counter()
    for menu, jumlah in pasangan:
        c[menu] += jumlah
    return list(c.items())

setelah = [combiner(h) for h in hasil_map]
print("\nPasangan dikirim tanpa combiner:", sum(len(h) for h in hasil_map))
print("Pasangan dikirim dengan combiner:", sum(len(h) for h in setelah))

map kasir A -> [('nasi goreng', 1), ('es teh', 2), ('mie ayam', 1), ('es teh', 1)]
map kasir B -> [('nasi goreng', 2), ('es teh', 1), ('es teh', 3), ('mie ayam', 1)]
map kasir C -> [('mie ayam', 2), ('nasi goreng', 1), ('es teh', 1), ('nasi goreng', 1)]

shuffle: {'nasi goreng': [1, 2, 1, 1], 'es teh': [2, 1, 1, 3, 1], 'mie ayam': [1, 1, 2]}

reduce: {'nasi goreng': 5, 'es teh': 8, 'mie ayam': 4} | total porsi: 17

Pasangan dikirim tanpa combiner: 12
Pasangan dikirim dengan combiner: 9


**H-9. Word Count Terdistribusi: Partisi sebagai Input Split**

Penerapan kerangka kerja MapReduce skala riil dilakukan pada keseluruhan 90 ribu judul berita, di mana setiap file partisi Parquet diperlakukan sebagai satu Input Split tersendiri bagi masing-masing mapper. Fungsi mapper_split membaca file Parquet, melakukan tokenisasi teks judul berita menjadi daftar kata, dan memancarkan pasangan tuple (kata, 1). Tahap shuffle mengagregasi frekuensi kemunculan kata secara global, sementara pemfilteran menggunakan kamus STOPWORD menyaring kata-kata fungsi umum untuk mendapatkan wawasan tematik yang bermakna.

In [13]:
STOPWORD = set("""yang di dan ke dari ini itu untuk dengan pada adalah akan tak tidak jadi soal
saat ada bisa juga oleh atau karena sebagai dalam usai hingga jika saja para lebih hanya
sudah belum masih kini lagi pun serta agar bakal dua tiga kok nih ya yg dgn""".split())

def tokenisasi(teks):
    return re.findall(r"[a-z]+", teks.lower())

# Input split = satu file partisi Parquet (satu kategori)
splits = sorted(Path(TERKURASI).rglob("*.parquet"))
print("Jumlah input split:", len(splits))
for p in splits:
    n = len(pd.read_parquet(p, columns=["judul"]))
    print(f"  {p.parent.name:<16} {n:>6,} judul")

def mapper_split(path):
    """Satu mapper memproses satu split dan mengembalikan pasangan (kata, 1)."""
    judul = pd.read_parquet(path, columns=["judul"])["judul"]
    return [(kata, 1) for t in judul for kata in tokenisasi(t)]

hasil_map = [mapper_split(p) for p in splits]
total_pasangan = sum(len(h) for h in hasil_map)

shuffle = collections.defaultdict(list)
for h in hasil_map:
    for kata, n in h:
        shuffle[kata].append(n)

hasil_semua = {k: sum(v) for k, v in shuffle.items()}

print("\nPasangan dari semua mapper:", f"{total_pasangan:,}")
print("Kata unik (key setelah shuffle):", f"{len(hasil_semua):,}")
print("\n10 kata teratas (semua kata):")
for k, n in collections.Counter(hasil_semua).most_common(10):
    print(f"  {k:<10} {n:>7,}")

top = [(k, n) for k, n in collections.Counter(hasil_semua).most_common() if k not in STOPWORD]
print("\n10 kata teratas (tanpa stopword):")
for k, n in top[:10]:
    print(f"  {k:<10} {n:>7,}")

# Cek silang dengan cara biasa
cek = collections.Counter(kata for t in bersih["judul"] for kata in tokenisasi(t))
print("\nSama dengan Counter biasa:", dict(cek) == hasil_semua)

Jumlah input split: 9
  kategori=finance 14,141 judul
  kategori=food     4,314 judul
  kategori=health   4,914 judul
  kategori=hot     16,290 judul
  kategori=inet     5,577 judul
  kategori=news    32,321 judul
  kategori=oto      4,375 judul
  kategori=sport    2,433 judul
  kategori=travel   6,453 judul

Pasangan dari semua mapper: 825,601
Kata unik (key setelah shuffle): 32,275

10 kata teratas (semua kata):
  di          27,390
  corona      17,239
  ini         10,124
  yang         8,668
  dan          6,785
  ke           5,592
  tak          5,096
  dari         4,697
  jadi         4,486
  untuk        3,723

10 kata teratas (tanpa stopword):
  corona      17,239
  rp           3,154
  virus        3,075
  kasus        2,743
  polisi       2,732
  positif      2,682
  indonesia    2,504
  warga        2,495
  rumah        2,465
  jokowi       2,317

Sama dengan Counter biasa: True


**H-10. Combiner dan Jumlah Data yang Berpindah
Python**

Cell ini menggunakan modul multiprocessing Python untuk mengukur kinerja eksekusi paralel dan membandingkan dampak penggunaan fungsi Combiner terhadap volume data yang ditransmisikan. Hasil eksperimen membuktikan bahwa penerapan combiner (menggunakan objek collections.Counter di sisi lokal mapper) mampu memangkas jumlah pasangan data secara drastis (hingga belasan kali lipat), menunjukkan bahwa dalam komputasi terdistribusi, biaya terbesar terletak pada perpindahan data (shuffle network cost), bukan sekadar jumlah prosesor CPU yang tersedia.

In [14]:
from multiprocessing import Pool

def mapper_combiner(path):
    """Mapper + combiner: hitung dulu di sisi mapper."""
    judul = pd.read_parquet(path, columns=["judul"])["judul"]
    c = collections.Counter()
    for t in judul:
        c.update(tokenisasi(t))
    return c

def jalankan(fn, daftar, proses):
    t0 = time.perf_counter()
    if proses == 1:
        hasil = [fn(x) for x in daftar]
    else:
        with Pool(proses) as pool:
            hasil = pool.map(fn, daftar)
    return hasil, time.perf_counter() - t0

print("CPU tersedia:", os.cpu_count())
print(f"{'skenario': <28}{'waktu (s)':>10}{'pasangan dikirim':>20}")

for proses in (1, 2):
    waktu = []
    for _ in range(3):
        h, w = jalankan(mapper_split, splits, proses); waktu.append(w)
    print(f"{proses} proses, tanpa combiner".ljust(28), f"{sorted(waktu)[1]:>10.3f}", f"{sum(len(x) for x in h):>20,}")

    waktu = []
    for _ in range(3):
        h2, w = jalankan(mapper_combiner, splits, proses); waktu.append(w)
    print(f"{proses} proses, dengan combiner".ljust(28), f"{sorted(waktu)[1]:>10.3f}", f"{sum(len(x) for x in h2):>20,}")

total = collections.Counter()
for c in h2:
    total.update(c)
print("\nHasil dengan combiner sama persis:", dict(total) == hasil_semua)

CPU tersedia: 2
skenario                     waktu (s)    pasangan dikirim
1 proses, tanpa combiner          1.520              825,601
1 proses, dengan combiner         0.579               74,009
2 proses, tanpa combiner          1.280              825,601
2 proses, dengan combiner         0.677               74,009

Hasil dengan combiner sama persis: True


**H-11. Data Skew dan Beban Reducer**

Fungsi partitioner menggunakan algoritma hash standar (zlib.crc32) untuk menentukan ke reducer mana sebuah kunci kata harus dilemparkan secara deterministik. Analisis ini mengevaluasi fenomena Data Skew (ketimpangan beban kerja antar reducer akibat dominasi kata kunci tertentu yang frekuensinya sangat tinggi seperti kata "di" atau "corona"). Hasil analisis menunjukkan bahwa sekadar menambah jumlah reducer tidak otomatis menyelesaikan masalah skew jika ketimpangan berasal langsung dari karakteristik distribusi data sumber itu sendiri.

In [15]:
R = 4  # jumlah reducer

def partitioner(kata, r=R):
    return zlib.crc32(kata.encode()) % r

def beban(pasangan_per_mapper, r=R):
    b = [0] * r
    for h in pasangan_per_mapper:
        for kata, n in h:
            b[partitioner(kata, r)] += 1
    return b

def tampil(nama, b):
    tot = sum(b)
    print(f"  {nama:<22}", "  ".join(f"R{i}={x:>7,} ({x/tot*100:4.1f}%)" for i, x in enumerate(b)))

tampil("tanpa combiner", beban(hasil_map))
tampil("dengan combiner", beban([list(c.items()) for c in h2]))

print("\nBeban reducer terberat dibanding bagian yang adil (tanpa combiner):")
for r in (4, 16, 64):
    b = beban(hasil_map, r)
    print(f"  {r:>3} reducer: terberat {max(b)/sum(b)*100:5.2f}% | bagian adil {100/r:5.2f}%")

print("\nBeban satu key (tanpa combiner):")
tot = sum(hasil_semua.values())
for kata, n in collections.Counter(hasil_semua).most_common(5):
    print(f"  {kata: <8} {n:>7,} pasangan = {n/tot*100:4.1f}% dari total, di reducer R{partitioner(kata)}")

  tanpa combiner         R0=240,836 (29.2%)  R1=170,066 (20.6%)  R2=190,727 (23.1%)  R3=223,972 (27.1%)
  dengan combiner        R0= 18,301 (24.7%)  R1= 18,267 (24.7%)  R2= 18,530 (25.0%)  R3= 18,911 (25.6%)

Beban reducer terberat dibanding bagian yang adil (tanpa combiner):
    4 reducer: terberat 29.17% | bagian adil 25.00%
   16 reducer: terberat  8.82% | bagian adil  6.25%
   64 reducer: terberat  4.28% | bagian adil  1.56%

Beban satu key (tanpa combiner):
  di        27,390 pasangan =  3.3% dari total, di reducer R0
  corona    17,239 pasangan =  2.1% dari total, di reducer R2
  ini       10,124 pasangan =  1.2% dari total, di reducer R0
  yang       8,668 pasangan =  1.0% dari total, di reducer R3
  dan        6,785 pasangan =  0.8% dari total, di reducer R3


**H-12. Hadoop Streaming**

Simulasi mekanistik lingkungan eksekusi Hadoop Streaming dilakukan dengan menulis skrip mapper.py dan reducer.py mandiri yang berkomunikasi melalui standar masukan (stdin) dan keluaran (stdout). Perintah pipe Linux (cat masukan.tsv | mapper | sort | reducer) mendemonstrasikan bagaimana utilitas sistem operasi sort bertindak sebagai tahap shuffle yang mengurutkan data berbasis byte (LC_ALL=C) sebelum diserahkan ke reducer, membuktikan kesamaan hasil logika pemrosesan terdistribusi dengan versi Python murni sebelumnya.

In [16]:
# Menulis file masukan dan script python untuk Hadoop Streaming
Path("masukan.tsv").write_text(
    "\n".join(f"{k}\t{j}" for k, j in zip(bersih["kategori"], bersih["judul"])) + "\n",
    encoding="utf-8"
)

Path("mapper.py").write_text('''import sys, re
for baris in sys.stdin:
    kategori, judul = baris.rstrip("\\n").split("\\t", 1)
    for kata in re.findall(r"[a-z]+", judul.lower()):
        print(f"{kata}\\t1")
''')

Path("reducer.py").write_text('''import sys
kunci_sekarang, total = None, 0
for baris in sys.stdin:
    kunci, nilai = baris.rstrip("\\n").split("\\t")
    if kunci != kunci_sekarang:
        if kunci_sekarang is not None:
            print(f"{kunci_sekarang}\\t{total}")
        kunci_sekarang, total = kunci, 0
    total += int(nilai)
if kunci_sekarang is not None:
    print(f"{kunci_sekarang}\\t{total}")
''')

print("Baris masukan:", sum(1 for _ in open("masukan.tsv", encoding="utf-8")))

Baris masukan: 90818


In [17]:
!head -n 3 masukan.tsv | python3 mapper.py
!time (cat masukan.tsv | python3 mapper.py | LC_ALL=C sort | python3 reducer.py > hasil_pipe.txt)
!wc -l hasil_pipe.txt
!sort -t$'\t' -k2,2nr hasil_pipe.txt | head -n 5

kemnaker	1
awasi	1
tka	1
di	1
meikarta	1
bni	1
digitalkan	1
bni	1
java	1
jazz	1
terbang	1
ke	1
australia	1
edhy	1
prabowo	1
mau	1
genjot	1
budi	1
daya	1
lobster	1

real	0m1.880s
user	0m2.028s
sys	0m0.096s
32275 hasil_pipe.txt
di	27390
corona	17239
ini	10124
yang	8668
dan	6785


In [18]:
hasil_pipe = {}
for baris in open("hasil_pipe.txt", encoding="utf-8"):
 kata, n = baris.rstrip("\n").split("\t")
 hasil_pipe[kata] = int(n)
print("Hasil pipe sama dengan hasil H-9:", hasil_pipe == hasil_semua)

Hasil pipe sama dengan hasil H-9: True


**H-13. Kata Teratas per Kategori, Ekspor ala Sqoop, dan Rangkuman**

Tahap akhir pipeline ini memperluas kunci analisis menjadi pasangan gabungan kategori dan kata (kategori, kata) guna mengidentifikasi karakteristik topik spesifik di setiap rubrik berita. Setelah agregasi selesai, hasil pemrosesan diekspor ke format Parquet dan disimpan ke dalam basis data relasional SQLite (laporan.db) melalui fungsi to_sql, yang menyimulasikan fungsionalitas Apache Sqoop Export untuk keperluan pelaporan analitik hilir (serving layer).

In [20]:
def mapper_kategori(path):
    """Key gabungan (kategori, kata); combiner dipasang di mapper."""
    df_ = pd.read_parquet(path, columns=["judul"])
    kategori = path.parent.name.split("=")[1]
    c = collections.Counter()
    for t in df_["judul"]:
        for kata in tokenisasi(t):
            if kata not in STOPWORD:
                c[(kategori, kata)] += 1
    return c

gabung = collections.Counter()
for c in map(mapper_kategori, splits):
    gabung.update(c)

per_kategori = collections.defaultdict(list)
for (kat, kata), n in gabung.items():
    per_kategori[kat].append((n, kata))

for kat in sorted(per_kategori):
    top5 = sorted(per_kategori[kat], reverse=True)[:5]
    print(f"{kat:<8}", ", ".join(f"{k} ({n:,})" for n, k in top5))

import sqlite3

SIAP = Path("siap_pakai")
SIAP.mkdir(exist_ok=True)

baris = [(kat, kata, n) for (kat, kata), n in gabung.items()]
hasil_df = pd.DataFrame(baris, columns=["kategori", "kata", "jumlah"])
hasil_df.to_parquet(SIAP / "kata_per_kategori.parquet", index=False)

# "sqoop export": pindahkan hasil ke database relasional
db = sqlite3.connect(SIAP / "laporan.db")
hasil_df.to_sql("kata_per_kategori", db, if_exists="replace", index=False)

print("Baris diekspor:", db.execute("SELECT COUNT(*) FROM kata_per_kategori").fetchone()[0])
print(db.execute("""SELECT kata, jumlah FROM kata_per_kategori
WHERE kategori = 'health' ORDER BY jumlah DESC LIMIT 3""").fetchall())
db.close()

finance  corona (2,470), rp (1,385), ri (1,125), harga (701), jokowi (642)
food     makanan (699), makan (539), foto (458), galeri (438), enak (358)
health   corona (2,590), virus (1,040), pasien (471), indonesia (427), meninggal (413)
hot      corona (1,658), anak (793), rumah (561), meninggal (545), the (528)
inet     corona (845), virus (341), ponsel (309), bikin (253), indonesia (249)
news     corona (7,539), polisi (2,244), kasus (2,015), warga (1,972), positif (1,876)
oto      mobil (1,205), corona (576), motor (432), rp (267), indonesia (256)
sport    corona (228), indonesia (223), motogp (209), masters (127), olimpiade (124)
travel   corona (1,023), wisata (525), foto (333), dunia (299), turis (284)
Baris diekspor: 73601
[('corona', 2590), ('virus', 1040), ('pasien', 471)]


In [21]:
import os
import shutil
from pathlib import Path
from google.colab import drive


drive.mount("/content/drive")

drive_path = "/content/drive/MyDrive/BigData/Praktikum5"
os.makedirs(drive_path, exist_ok=True)

folder_hasil = ["lapisan_mentah", "lapisan_terkurasi", "siap_pakai"]

for nama in folder_hasil:
  sumber = Path(nama)
  tujuan = Path(drive_path) / nama
  if sumber.exists():
    if tujuan.exists():
      shutil.rmtree(tujuan)
    shutil.copytree(sumber, tujuan)
    print(f"Berhasil menyalin '{nama}' ke Google Drive!")
  else:
    print(f"Peringatan: Folder lokal '{nama}' tidak ditemukan.")

print("Selesai! Semua file aman di folder target:", drive_path)

Mounted at /content/drive
Berhasil menyalin 'lapisan_mentah' ke Google Drive!
Berhasil menyalin 'lapisan_terkurasi' ke Google Drive!
Berhasil menyalin 'siap_pakai' ke Google Drive!
Selesai! Semua file aman di folder target: /content/drive/MyDrive/BigData/Praktikum5


## **Latihan**

**Latihan 1: Stopword Domain**

Latihan ini memodifikasi kamus STOPWORD global dengan menyisipkan kata-kata fungsi atau preposisi tambahan yang sering muncul namun tidak membawa makna tematik yang unik. Dengan menjalankan ulang fungsi mapper_kategori, kita dapat mengamati bagaimana peringkat lima kata teratas pada setiap rubrik berita bergeser karena dominasi kata-kata noise tersebut berhasil disaring.

In [22]:
STOPWORD.update(["juga", "pada", "untuk", "oleh", "karena"])

# jalankan ulang agregasi H-13 u, melihat perubahan kata teratas per kategori
gabung_l1 = collections.Counter()
for c in map(mapper_kategori, splits):
  gabung_l1.update(c)

per_kategori_l1 = collections.defaultdict(list)
for (kat, kata), n in gabung_l1.items():
  per_kategori_l1[kat].append((n, kata))

print("Hasil Top 5 per Kategori setelah Penambahan Stopword Domain:")
for kat in sorted(per_kategori_l1):
  top5 = sorted(per_kategori_l1[kat], reverse=True)[:5]
  print(f"{kat:<8} :", ", ".join(f"{k} ({n:,})" for n, k in top5))

Hasil Top 5 per Kategori setelah Penambahan Stopword Domain:
finance  : corona (2,470), rp (1,385), ri (1,125), harga (701), jokowi (642)
food     : makanan (699), makan (539), foto (458), galeri (438), enak (358)
health   : corona (2,590), virus (1,040), pasien (471), indonesia (427), meninggal (413)
hot      : corona (1,658), anak (793), rumah (561), meninggal (545), the (528)
inet     : corona (845), virus (341), ponsel (309), bikin (253), indonesia (249)
news     : corona (7,539), polisi (2,244), kasus (2,015), warga (1,972), positif (1,876)
oto      : mobil (1,205), corona (576), motor (432), rp (267), indonesia (256)
sport    : corona (228), indonesia (223), motogp (209), masters (127), olimpiade (124)
travel   : corona (1,023), wisata (525), foto (333), dunia (299), turis (284)


**Latihan 2: Analisis Per Bulan**

Fungsi mapper_kategori_per_bulan mengubah struktur key agregasi dari yang sebelumnya berbasis kategori menjadi format tuple (bulan, kata). Dengan mengekstrak informasi bulan dari kolom tanggal artikel, proses MapReduce dapat melacak pergeseran tren tematik bulanan—misalnya, bagaimana fokus kata teratas bergeser dari isu umum di bulan Januari (2020-01) menuju puncak eskalasi pandemi di bulan-bulan berikutnya seperti Mei (2020-05).

In [23]:
def mapper_kategori_per_bulan(path):
  """Mapper dengan key gabungan (bulan, kata)."""
  df_ = pd.read_parquet(path, columns=["judul", "tanggal"])
  df_["bulan"] = pd.to_datetime(df_["tanggal"]).dt.strftime("%Y-%m")
  c = collections.Counter()
  for _, row in df_.iterrows():
    bln = row["bulan"]
    for kata in tokenisasi(row["judul"]):
      if kata not in STOPWORD:
        c[(bln, kata)] += 1
  return c


# Agregasi data berdasarkan bulan dan kata
gabung_bulan = collections.Counter()
for c in map(mapper_kategori_per_bulan, splits):
  gabung_bulan.update(c)

# Tampilkan kata teratas untuk bulan Januari (2020-01) dan Mei (2020-05)
for b in ["2020-01", "2020-05"]:
  top_b = [
      (kt, val) for (bln, kt), val in gabung_bulan.most_common() if bln == b
  ]
  print(f"Top 5 Kata Bulan {b}:", top_b[:5])

Top 5 Kata Bulan 2020-01: [('banjir', 1220), ('rp', 616), ('polisi', 554), ('jokowi', 516), ('tahun', 493)]
Top 5 Kata Bulan 2020-05: [('corona', 2935), ('kasus', 640), ('positif', 628), ('pandemi', 596), ('warga', 565)]


**Latihan 3: Ekstraksi Bigram**

Latihan ini memperluas unit analisis dari kata tunggal (unigram) menjadi pasangan dua kata berurutan (bigram), seperti frasa "virus corona". Fungsi tokenisasi_bigram menyaring pasangan kata yang tidak mengandung stopword. Di sini terlihat jelas fungsi combiner lokal dalam merangkum frekuensi frasa yang sama di dalam satu split file Parquet sebelum ditransmisikan ke tahap berikutnya.

In [24]:
def tokenisasi_bigram(teks):
  """Menghasilkan pasangan dua kata berurutan (bigram)."""
  kata2 = re.findall(r"[a-z]+", teks.lower())
  bigram_list = []
  for i in range(len(kata2) - 1):
    k1, k2 = kata2[i], kata2[i + 1]
    if k1 not in STOPWORD and k2 not in STOPWORD:
      bigram_list.append(f"{k1} {k2}")
  return bigram_list


def mapper_bigram(path):
  df_ = pd.read_parquet(path, columns=["judul"])
  c = collections.Counter()
  for t in df_["judul"]:
    for bg in tokenisasi_bigram(t):
      c[bg] += 1
  return c


# Eksekusi MapReduce untuk Bigram
hasil_map_bigram = [mapper_bigram(p) for p in splits]
total_pasangan_bigram = sum(
    sum(h.values()) for h in hasil_map_bigram
)  # total frekuensi

# Kombinasi dengan combiner (menjumlahkan di sisi mapper)
combiner_bigram = [dict(h) for h in hasil_map_bigram]
total_unik_bigram_mapper = sum(len(h) for h in combiner_bigram)

print(f"Total kemunculan bigram (sebelum combiner/mentah): {total_pasangan_bigram:,}")
print(f"Jumlah pasangan unik dikirim (setelah combiner lokal): {total_unik_bigram_mapper:,}")

Total kemunculan bigram (sebelum combiner/mentah): 518,936
Jumlah pasangan unik dikirim (setelah combiner lokal): 309,305


**Latihan 4: Variasi Jumlah Reducer**

memodifikasi parameter jumlah reducer menjadi 2, 8, dan 32 menggunakan fungsi beban yang telah didefinisikan sebelumnya di H-11. Eksperimen ini membuktikan bahwa semakin besar jumlah reducer yang digunakan, porsi persentase beban pada reducer terberat memang akan menurun, namun tetap tidak akan pernah bisa mencapai titik ideal bagian yang adil (1/R) secara merata kare

In [25]:
# Uji variasi jumlah reducer R = 2, 8, dan 32 pada data H-11
for r_val in (2, 8, 32):
  b = beban(hasil_map, r_val)
  tot_b = sum(b)
  max_beban = max(b)
  bagian_adil = 100 / r_val
  persen_terberat = (max_beban / tot_b) * 100
  print(
      f"R = {r_val:>2} reducer -> Reducer terberat: {persen_terberat:5.2f}%"
      f" | Bagian ideal/adil: {bagian_adil:5.2f}%"
  )

R =  2 reducer -> Reducer terberat: 52.27% | Bagian ideal/adil: 50.00%
R =  8 reducer -> Reducer terberat: 16.83% | Bagian ideal/adil: 12.50%
R = 32 reducer -> Reducer terberat:  5.82% | Bagian ideal/adil:  3.12%


**Latihan 5: Estimasi Waktu Crawling Berdasarkan Jeda (Delay)**

Menganalisis dampak parameter jeda waktu (delay) terhadap durasi total proses crawling web. Berdasarkan perhitungan teoretis, penerapan jeda kecil (misalnya 0,05 detik untuk 50 halaman pertama) mempercepat proses pengambilan data secara signifikan, namun berisiko membebani server target. Sebaliknya, penggunaan jeda standar 1 detik untuk seluruh 911 halaman memerlukan waktu tunggu murni sekitar 911 detik (15 menit) di luar latensi jaringan, demi menjaga etika akses dan menghindari pemblokiran IP oleh server.  

In [26]:
# Simulasi perhitungan teoretis waktu crawl
halaman_total = 911
halaman_uji = 50
jeda_uji = 0.05  # detik
jeda_standar = 1.0  # detik

# Estimasi waktu tunggu jeda murni
waktu_jeda_50 = halaman_uji * jeda_uji
waktu_jeda_911 = halaman_total * jeda_standar

print(f"Estimasi waktu jeda untuk {halaman_uji} halaman (jeda {jeda_uji}s): {waktu_jeda_50:.1f} detik")
print(
    f"Estimasi waktu jeda untuk {halaman_total} halaman (jeda {jeda_standar}s):"
    f" {waktu_jeda_911 / 60:.1f} menit ({waktu_jeda_911} detik)"
)

Estimasi waktu jeda untuk 50 halaman (jeda 0.05s): 2.5 detik
Estimasi waktu jeda untuk 911 halaman (jeda 1.0s): 15.2 menit (911.0 detik)


**Latihan 6: Integrasi Sitemap Alternatif & Uji Kualitas**

menguji fleksibilitas pipeline ingestion dengan mengarahkan target URL ke indeks sitemap alternatif dari bagian lain situs detik.com. Setelah 20 judul berita berhasil di-crawl dari sumber baru tersebut, aturan kualitas H-6 (aturan_kualitas) diterapkan kembali untuk memverifikasi integritas data, mendeteksi ada atau tidaknya judul kosong, serta memeriksa distribusi kategori yang muncul.   

In [27]:
# Mengganti salah satu sitemap dalam list SITEMAP_DIPILIH dengan sitemap lain
SITEMAP_ALTERNATIF = [
    "https://news.detik.com/indeks/sitemap_news.xml"  # contoh sitemap alternatif
]


# Fungsi uji kualitas pada 20 judul hasil crawl sitemap baru
def uji_kualitas_sitemap_baru(sampel_baru):
  df_smp = pd.DataFrame(sampel_baru)
  if df_smp.empty:
    print("Sampel kosong atau gagal di-crawl.")
    return
  print("Jumlah judul terkumpul:", len(df_smp))
  print("Judul kosong:", (df_smp["judul"].str.strip() == "").sum())
  print("URL duplikat:", df_smp["url"].duplicated().sum())
  print("Distribusi Kategori:", df_smp["kategori"].value_counts().to_dict())

**Latihan 7: Analisis Frekuensi Kata Kunci (Keywords)**

memanfaatkan metadata kolom kata_kunci yang disediakan secara langsung oleh struktur sitemap berita. Melalui fungsi mapper_keyword_sitemap, sistem memecah string kata kunci yang dipisahkan oleh tanda koma, membersihkan spasi (strip), dan melakukan agregasi frekuensi global menggunakan Collections.Counter. Hasilnya memperlihatkan daftar kata kunci metadata redaksional yang paling sering disematkan pada rubrik-rubrik berita tersebut

In [30]:
# Ekstraksi kata kunci khusus dari file hasil sitemap H-4 (yang punya kolom kata_kunci)
from pathlib import Path

path_sitemap_jsonl = Path("contoh_crawl/detik_sitemap.jsonl")

if path_sitemap_jsonl.exists():
  df_sitemap_smp = pd.read_json(path_sitemap_jsonl, lines=True)
  kw_counter = collections.Counter()

  for kw_list in df_sitemap_smp.get("kata_kunci", []):
    if pd.notna(kw_list) and kw_list.strip() != "":
      for kw in str(kw_list).split(","):
        clean_kw = kw.strip().lower()
        if clean_kw:
          kw_counter[clean_kw] += 1

  if len(kw_counter) > 0:
    print("10 Kata Kunci Terbanyak dari Metadata Sitemap H-4:")
    for kw, count in kw_counter.most_common(10):
      print(f" - {kw:<25} : {count:,} kemunculan")
  else:
    print("Kolom kata_kunci pada sampel sitemap kosong.")
else:
  print("File sitemap JSONL H-4 belum ditemukan. Jalankan dulu cell H-4!")

10 Kata Kunci Terbanyak dari Metadata Sitemap H-4:
 - narkoba                   : 2 kemunculan
 - bnn                       : 2 kemunculan
 - polri                     : 2 kemunculan
 - telkom                    : 2 kemunculan
 - nvidia                    : 2 kemunculan
 - jensen huang              : 2 kemunculan
 - basket                    : 2 kemunculan
 - ibl all indonesian 2026   : 2 kemunculan
 - ibl all indonesian        : 2 kemunculan
 - muzammil                  : 1 kemunculan


##**Tugas Mandiri**

Implementasi Inverted Index ini dibuat dengan paradigma MapReduce berbasis Python untuk memetakan setiap kata unik langsung ke daftar ID dokumen (posting list) yang memuat kata tersebut. Alur kerjanya meliputi:

Tahap Tokenisasi & Stopword Removal yang dimana setiap baris judul berita pada partisi data Parquet (splits) diekstrak, diubah menjadi huruf kecil (lowercase), dan dipecah menjadi kata-kata tunggal (unigram). Kata-kata umum yang tidak membawa informasi tematik penting disaring menggunakan kamus STOPWORD.

Melakukan penanganan Data Khusus (Kategori Inet) untuk baris data dari kategori tertentu (seperti rubrik inet) di mana kolom ID dokumen bernilai kosong atau rusak, sistem secara otomatis memberikan penanda tiruan berbasis string yang unik (misal: MIS-i-idx). Hal ini dilakukan agar data dari kategori tersebut tidak terhapus dan tetap terindeks tanpa merusak integritas tipe data integer pada dokumen lainnya.

Lalu tahap Shuffle & Reduce dimna seluruh pasangan (kata, id) dikelompokkan ke dalam struktur data hash map menggunakan tipe Set guna memastikan ID dokumen bersifat unik per kata (menghindari duplikat jika sebuah kata muncul berulang kali di judul yang sama). Terakhir, daftar ID diurutkan secara menaik (ascending) untuk memudahkan proses pencarian teks.

In [35]:
import collections
import re

# Pastikan stopword standar sudah ada
STOPWORD = {
    'dan',
    'di',
    'dari',
    'ke',
    'yang',
    'untuk',
    'pada',
    'adalah',
    'sebagai',
    'ini',
    'itu',
    'atau',
    'dengan',
    'dalam',
    'oleh',
    'karena',
    'tersebut',
}


def tokenisasi(teks):
  if not isinstance(teks, str):
    return []
  return re.findall(r'[a-z]+', teks.lower())


# Membangun Inverted Index langsung dari list 'splits' partisi parquet yang aktif di memori
semua_pasangan_ii = []

for i, p in enumerate(splits):
  try:
    df_part = pd.read_parquet(p)
    for idx, row in df_part.iterrows():
      # Ambil id jika ada, atau buat id tiruan jika kosong
      aid = row.get('id', None)
      if pd.isna(aid):
        aid = f'MIS-{i}-{idx}'
      else:
        aid = int(aid)

      judul_teks = row.get('judul', '')
      for kata in tokenisasi(judul_teks):
        if kata not in STOPWORD:
          semua_pasangan_ii.append((kata, aid))
  except Exception as e:
    continue

# Shuffle & Reduce
inverted_index = collections.defaultdict(set)
for kata, aid in semua_pasangan_ii:
  inverted_index[kata].add(aid)

inverted_index_final = {
    kata: sorted(list(ids), key=lambda x: str(x))
    for kata, ids in inverted_index.items()
}

print(f'Total kata unik dalam Inverted Index: {len(inverted_index_final):,}')
print('\nContoh hasil Inverted Index (3 kata pilihan beserta 5 ID pertamanya):')

contoh_kata = ['corona', 'jokowi', 'polisi']
for kw in contoh_kata:
  if kw in inverted_index_final:
    daftar_id = inverted_index_final[kw][:5]
    print(
        f" - Kata '{kw}': {daftar_id} ... (Total dokumen:"
        f' {len(inverted_index_final[kw]):,})'
    )
  else:
    print(f" - Kata '{kw}': Tidak ditemukan dalam indeks.")

Total kata unik dalam Inverted Index: 32,258

Contoh hasil Inverted Index (3 kata pilihan beserta 5 ID pertamanya):
 - Kata 'corona': [4856495, 4856598, 4868317, 4868545, 4869163] ... (Total dokumen: 17,228)
 - Kata 'jokowi': [4842674, 4842690, 4842703, 4842774, 4843215] ... (Total dokumen: 2,316)
 - Kata 'polisi': [4842273, 4842280, 4842541, 4842612, 4843641] ... (Total dokumen: 2,724)
